# EMG2QWERTY Project Experiments

This notebook is the working experiment notebook for the EMG2QWERTY final project.

## Purpose
- launch training runs
- evaluate checkpoints
- log experiment results
- keep concise debugging notes

## Project scope
Subject: **89335547**

Main goals:
1. Train the baseline
2. Compare different architectures
3. Study preprocessing / augmentation
4. Study number of channels vs CER
5. Study amount of training data vs CER
6. Study sampling rate vs CER

## Note
This notebook is for **running and logging experiments**.

A separate notebook, `project_results.ipynb`, will be used later for:
- clean summary tables
- report-ready plots
- final conclusions

# Setup

Imports packages, defines project paths, and creates folders for logs, results, plots, and experiment metadata.

In [1]:
import os
import json
import yaml
import random
import subprocess
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
from IPython.display import display, Markdown

In [2]:
PROJECT_ROOT = Path("/home/bzhao29/emg2qwerty")
CONFIG_MODEL_DIR = PROJECT_ROOT / "config" / "model"
PACKAGE_DIR = PROJECT_ROOT / "emg2qwerty"

NOTEBOOK_RESULTS_DIR = PROJECT_ROOT / "notebook_results"
TABLES_DIR = NOTEBOOK_RESULTS_DIR / "tables"
NOTES_DIR = NOTEBOOK_RESULTS_DIR / "notes"

RESULTS_CSV = TABLES_DIR / "master_results.csv"
EXPERIMENT_NOTES = NOTES_DIR / "project_experiment_notes.txt"

for d in [NOTEBOOK_RESULTS_DIR, TABLES_DIR, NOTES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Reproducibility helpers

In [3]:
SEED = 42

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

set_seed(SEED)

def timestamp():
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

def save_json(obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w") as f:
        json.dump(obj, f, indent=2)

def append_note(text):
    with open(EXPERIMENT_NOTES, "a") as f:
        f.write(f"\n[{timestamp()}]\n{text}\n")

print("Seed set to:", SEED)
print("Experiment notes file:", EXPERIMENT_NOTES)

Seed set to: 42
Experiment notes file: /home/bzhao29/emg2qwerty/notebook_results/notes/project_experiment_notes.txt


# Results table setup

This CSV is the single source of truth for all completed experiments.

In [4]:
RESULT_COLUMNS = [
    "experiment_name",
    "group",
    "model_type",
    "config_path",
    "checkpoint_path",
    "preprocessing",
    "augmentation",
    "num_channels",
    "train_fraction",
    "sampling_rate",
    "epochs",
    "val_CER",
    "test_CER",
    "DER",
    "IER",
    "SER",
    "notes",
    "status",
]

if RESULTS_CSV.exists():
    results_df = pd.read_csv(RESULTS_CSV)
else:
    results_df = pd.DataFrame(columns=RESULT_COLUMNS)
    results_df.to_csv(RESULTS_CSV, index=False)

results_df.tail(3)

,experiment_name,group,model_type,config_path,checkpoint_path,augmentation,num_channels,train_fraction,sampling_rate,epochs,val_CER,test_CER,DER,IER,SER,notes,status
43,tds_gru_sampling_1000Hz,sampling_rate,TDS+GRU,/home/bzhao29/emg2qwerty/config/model/tds_gru_...,/home/bzhao29/emg2qwerty/logs/2026-03-12/06-39...,band_rotation+temporal_jitter,32,1.0,1000,40,27.846699,21.547440,1.966717,5.684029,13.896693,"TDS+GRU setup, factor=2 (1000 Hz), 40 epochs.",done
44,tds_gru_sampling_500Hz,sampling_rate,TDS+GRU,/home/bzhao29/emg2qwerty/config/model/tds_gru_...,/home/bzhao29/emg2qwerty/logs/2026-03-12/06-11...,band_rotation+temporal_jitter,32,1.0,500,40,61.408951,46.833801,0.453858,29.954615,16.425329,"TDS+GRU setup, factor=4 (500 Hz), 40 epochs.",done
45,tds_gru_150epochs,training_length,TDS+GRU,/home/bzhao29/emg2qwerty/config/model/tds_gru_...,/home/bzhao29/emg2qwerty/logs/2026-03-12/07-12...,band_rotation+temporal_jitter,32,1.0,2000,150,12.051395,13.680571,1.296737,2.398962,9.984872,Extended training (150 epochs) with hidden_siz...,done


## Helper functions

- register experiments
- save generated configs
- append results rows
- find checkpoints
- parse metrics

In [5]:
def save_results_df():
    global results_df
    results_df.to_csv(RESULTS_CSV, index=False)

def add_or_replace_result(row):
    global results_df
    row_df = pd.DataFrame([row])
    if "experiment_name" not in row:
        raise ValueError("row must contain experiment_name")

    if len(results_df) > 0 and (results_df["experiment_name"] == row["experiment_name"]).any():
        results_df = results_df[results_df["experiment_name"] != row["experiment_name"]]

    results_df = pd.concat([results_df, row_df], ignore_index=True)
    save_results_df()

def show_completed_results():
    global results_df
    done = results_df[(results_df["status"] == "done") & (results_df["test_CER"].notna())].copy()
    if len(done) == 0:
        print("No completed results yet.")
    else:
        display(done.sort_values(["group", "test_CER"], na_position="last"))

## Shell command runner

Use this helper for training/evaluation commands so all commands are visible and easy to reuse.

In [6]:
def run_cmd(cmd, cwd=PROJECT_ROOT, capture_output=False):
    print("\n[Running command]")
    print(cmd)
    result = subprocess.run(
        cmd,
        shell=True,
        cwd=str(cwd),
        text=True,
        capture_output=capture_output
    )
    if capture_output:
        print(result.stdout)
        if result.stderr:
            print(result.stderr)
    if result.returncode != 0:
        raise RuntimeError(f"Command failed with return code {result.returncode}")
    return result

# Official baseline

This is the official baseline used as the comparison reference for all future experiments.

In [34]:
!LD_LIBRARY_PATH=$CONDA_PREFIX/lib python -m emg2qwerty.train \
user=single_user \
trainer.max_epochs=40 \
trainer.accelerator=gpu \
trainer.devices=1 \
num_workers=4 \
batch_size=16

[2026-03-08 07:52:41,061][__main__][INFO] - 
Config:
user: single_user
dataset:
  train:
  - user: 89335547
    session: 2021-06-03-1622765527-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-02-1622681518-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-04-1622863166-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-22-1627003020-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-21-1626916256-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-22-1627004019-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-05-1622885888-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-02-1622679967-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f

In [56]:
OFFICIAL_BASELINE = {
    "experiment_name": "baseline__tds__official",
    "group": "architecture",
    "model_type": "TDS",
    "config_path": "/home/bzhao29/emg2qwerty/config/model/tds_conv_ctc.yaml",
    "checkpoint_path": "/home/bzhao29/emg2qwerty/logs/2026-03-08/07-52-40/checkpoints/epoch=39-step=9600.ckpt",
    "preprocessing": "baseline_logspec",
    "augmentation": "None",
    "num_channels": 32,
    "train_fraction": 1.0,
    "sampling_rate": 2000,
    "epochs": 40,
    "val_CER": 19.5392127319336,
    "test_CER": 20.726173400878906,
    "DER": 1.8802679777145386,
    "IER": 4.4953532218933105,
    "SER": 14.350550651550293,
    "notes": "Official baseline result",
    "status": "done",
}

add_or_replace_result(OFFICIAL_BASELINE)

# TDS + BiLSTM hybrid module

This is the recurrent hybrid model.
It keeps the baseline TDS frontend/encoder and adds BiLSTM layers to capture temporal dependencies

In [13]:
!LD_LIBRARY_PATH=$CONDA_PREFIX/lib python -m emg2qwerty.train \
user=single_user \
model=tds_bilstm_bestaug \
trainer.max_epochs=80 \
optimizer.lr=0.0003 \
lr_scheduler.scheduler.warmup_epochs=12 \
module.hidden_size=384 \
module.num_layers=3 \
module.dropout=0.15 \
trainer.accelerator=gpu \
trainer.devices=1 \
num_workers=4 \
batch_size=16 \
+trainer.enable_progress_bar=false \
+trainer.log_every_n_steps=50

[2026-03-10 01:48:15,422][__main__][INFO] - 
Config:
user: single_user
dataset:
  train:
  - user: 89335547
    session: 2021-06-03-1622765527-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-02-1622681518-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-04-1622863166-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-22-1627003020-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-21-1626916256-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-22-1627004019-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-05-1622885888-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-02-1622679967-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f

In [57]:
TDS_BILSTM_TUNED_RESULT = {
    "experiment_name": "tds_bilstm",
    "group": "architecture",
    "model_type": "TDS+BiLSTM",
    "config_path": "/home/bzhao29/emg2qwerty/config/model/tds_bilstm_ctc_noaug.yaml",
    "checkpoint_path": "/home/bzhao29/emg2qwerty/logs/2026-03-10/01-48-15/checkpoints/epoch=73-step=17760.ckpt",
    "preprocessing": "baseline_logspec",
    "augmentation": "band_rotation+temporal_jitter",
    "num_channels": 32,
    "train_fraction": 1.0,
    "sampling_rate": 2000,
    "epochs": 80,
    "val_CER": 12.959680557250977,
    "test_CER": 14.80440902709961,
    "DER": 1.469634771347046,
    "IER": 2.0747785568237305,
    "SER": 11.259995460510254,
    "notes": "Tuned TDS+BiLSTM: hidden_size=384, num_layers=3, dropout=0.15, lr=3e-4, warmup=12, epochs=80. Best checkpoint at epoch 73.",
    "status": "done",
}

add_or_replace_result(TDS_BILSTM_TUNED_RESULT)

# TDS + GRU hybrid module

This model keeps the baseline TDS frontend/encoder and adds GRU layers for long-term dependencies

In [10]:
!LD_LIBRARY_PATH=$CONDA_PREFIX/lib python -m emg2qwerty.train \
user=single_user \
model=tds_gru_bestaug \
train=false \
checkpoint='"/home/bzhao29/emg2qwerty/logs/2026-03-09/23-20-56/checkpoints/epoch=77-step=18720.ckpt"' \
optimizer.lr=0.0003 \
lr_scheduler.scheduler.warmup_epochs=12 \
module.hidden_size=384 \
module.num_layers=3 \
module.dropout=0.15 \
trainer.accelerator=gpu \
trainer.devices=1 \
num_workers=4 \
batch_size=16 \
seed=42 \
+trainer.enable_progress_bar=false \
+trainer.log_every_n_steps=50

[2026-03-11 09:50:41,297][__main__][INFO] - 
Config:
user: single_user
dataset:
  train:
  - user: 89335547
    session: 2021-06-03-1622765527-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-02-1622681518-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-04-1622863166-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-22-1627003020-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-21-1626916256-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-22-1627004019-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-05-1622885888-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-02-1622679967-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f

In [58]:
TDS_GRU_TUNED_RESULT = {
    "experiment_name": "tds_gru_best",
    "group": "architecture",
    "model_type": "TDS+GRU",
    "config_path": "/home/bzhao29/emg2qwerty/config/model/tds_gru_ctc_best_q3.yaml",
    "checkpoint_path": "/home/bzhao29/emg2qwerty/logs/2026-03-09/23-20-56/checkpoints/epoch=77-step=18720.ckpt",
    "preprocessing": "baseline_logspec",
    "augmentation": "band_rotation+temporal_jitter",
    "num_channels": 32,
    "train_fraction": 1.0,
    "sampling_rate": 2000,
    "epochs": 80,
    "val_CER": 12.738147735595703,
    "test_CER": 13.594121932983398,
    "DER": 1.339961051940918,
    "IER": 2.4854116439819336,
    "SER": 9.76874828338623,
    "notes": "Best TDS+GRU setup: hidden_size=384, num_layers=3, dropout=0.15, lr=3e-4, warmup=12, epochs=80, with band rotation + temporal jitter.",
    "status": "done",
}

add_or_replace_result(TDS_GRU_TUNED_RESULT)

# TDS + Transformer module

TDS encoder with transformer layers using self-attention.

In [97]:
!LD_LIBRARY_PATH=$CONDA_PREFIX/lib python -m emg2qwerty.train \
user=single_user \
model=tds_transformer_bestaug \
trainer.max_epochs=80 \
optimizer.lr=0.0003 \
lr_scheduler.scheduler.warmup_epochs=12 \
trainer.accelerator=gpu \
module.dropout=0.2 \
trainer.devices=1 \
num_workers=8 \
batch_size=16 \
seed=42 \
+trainer.enable_progress_bar=false \
+trainer.log_every_n_steps=50

[2026-03-13 05:16:36,770][__main__][INFO] - 
Config:
user: single_user
dataset:
  train:
  - user: 89335547
    session: 2021-06-03-1622765527-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-02-1622681518-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-04-1622863166-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-22-1627003020-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-21-1626916256-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-22-1627004019-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-05-1622885888-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-02-1622679967-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f

In [7]:
TDS_Transformer_RESULT = {
    "experiment_name": "tds_transformer",
    "group": "architecture",
    "model_type": "TDS+Transformer",
    "config_path": "/home/bzhao29/emg2qwerty/config/model/tds_transformer_bestaug.yaml",
    "checkpoint_path": "/home/bzhao29/emg2qwerty/logs/2026-03-13/05-16-36/checkpoints/epoch=59-step=14400.ckpt",
    "preprocessing": "baseline_logspec",
    "augmentation": "band_rotation+temporal_jitter",
    "num_channels": 32,
    "train_fraction": 1.0,
    "sampling_rate": 2000,
    "epochs": 80,
    "val_CER": 15.640230178833008,
    "test_CER": 29.28463363647461,
    "DER": 5.316619873046875,
    "IER": 8.428787231445312,
    "SER": 15.539226531982422,
    "notes": "TDS+Transformer setup: hidden_size=384, num_layers=3, dropout=0.2, lr=3e-4, warmup=12, epochs=80, with band rotation + temporal jitter.",
    "status": "done",
}

add_or_replace_result(TDS_Transformer_RESULT)

# TDS + Conformer Module

In [4]:
!LD_LIBRARY_PATH=$CONDA_PREFIX/lib python -m emg2qwerty.train \
user=single_user \
model=tds_conformer_bestaug \
trainer.max_epochs=80 \
optimizer.lr=0.0003 \
lr_scheduler.scheduler.warmup_epochs=12 \
trainer.accelerator=gpu \
module.dropout=0.2 \
trainer.devices=1 \
num_workers=8 \
batch_size=16 \
+trainer.enable_progress_bar=false \
+trainer.log_every_n_steps=50

[2026-03-13 01:38:09,233][__main__][INFO] - 
Config:
user: single_user
dataset:
  train:
  - user: 89335547
    session: 2021-06-03-1622765527-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-02-1622681518-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-04-1622863166-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-22-1627003020-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-21-1626916256-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-22-1627004019-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-05-1622885888-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-02-1622679967-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f

In [8]:
TDS_CONFORMER_RESULT = {
    "experiment_name": "tds_conformer",
    "group": "architecture",
    "model_type": "TDS+Conformer",
    "config_path": "/home/bzhao29/emg2qwerty/config/model/tds_conformer_bestaug.yaml",
    "checkpoint_path": "/home/bzhao29/emg2qwerty/logs/2026-03-13/01-38-09/checkpoints/epoch=76-step=18480.ckpt",
    "preprocessing": "baseline_logspec",
    "augmentation": "band_rotation+temporal_jitter+specaug",
    "num_channels": 32,
    "train_fraction": 1.0,
    "sampling_rate": 2000,
    "epochs": 50,
    "val_CER": 13.934426307678223,
    "test_CER": 17.354658126831055,
    "DER": 2.463799476623535,
    "IER": 2.550248622894287,
    "SER": 12.340609550476074,
    "notes": "TDS+Conformer; input_dim=384, num_heads=4, ffn_dim=768, num_layers=4, depthwise_conv_kernel_size=31, dropout=0.15, lr=3e-4, warmup=12, epochs=80.",
    "status": "done",
}

add_or_replace_result(TDS_CONFORMER_RESULT)

# Experimentation with Data-preprocessing and Data-augmentation

Using our best model: TDS + GRU model, we are going to test various data processing techniques to see if we can decrease CER further

### No Augmentation

In [8]:
!LD_LIBRARY_PATH=$CONDA_PREFIX/lib python -m emg2qwerty.train \
user=single_user \
model=tds_gru_ctc_aug_none \
trainer.max_epochs=80 \
optimizer.lr=0.0003 \
lr_scheduler.scheduler.warmup_epochs=12 \
trainer.accelerator=gpu \
trainer.devices=1 \
num_workers=4 \
batch_size=16 \
+trainer.enable_progress_bar=false \
+trainer.log_every_n_steps=50

[2026-03-10 17:46:01,641][__main__][INFO] - 
Config:
user: single_user
dataset:
  train:
  - user: 89335547
    session: 2021-06-03-1622765527-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-02-1622681518-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-04-1622863166-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-22-1627003020-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-21-1626916256-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-22-1627004019-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-05-1622885888-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-02-1622679967-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f

In [59]:
TDS_GRU_AUG_NONE_RESULT = {
    "experiment_name": "tds_gru_aug_none",
    "group": "augmentation",
    "model_type": "TDS+GRU",
    "config_path": "/home/bzhao29/emg2qwerty/config/model/tds_gru_ctc_aug_none.yaml",
    "checkpoint_path": "/home/bzhao29/emg2qwerty/logs/2026-03-10/17-46-01/checkpoints/epoch=77-step=18720.ckpt",
    "preprocessing": "baseline_logspec",
    "augmentation": "none",
    "num_channels": 32,
    "train_fraction": 1.0,
    "sampling_rate": 2000,
    "epochs": 80,
    "val_CER": 14.155959129333496,
    "test_CER": 16.425329208374023,
    "DER": 1.7073698043823242,
    "IER": 2.7015345096588135,
    "SER": 12.016425132751465,
    "notes": "TDS+GRU with no augmentation; hidden_size=384, num_layers=3, dropout=0.15, lr=3e-4, warmup=12, epochs=80",
    "status": "done",
}

add_or_replace_result(TDS_GRU_AUG_NONE_RESULT)

### Band rotation only

In [9]:
!LD_LIBRARY_PATH=$CONDA_PREFIX/lib python -m emg2qwerty.train \
user=single_user \
model=tds_gru_ctc_aug_bandrot \
trainer.max_epochs=80 \
optimizer.lr=0.0003 \
lr_scheduler.scheduler.warmup_epochs=12 \
trainer.accelerator=gpu \
trainer.devices=1 \
num_workers=4 \
batch_size=16 \
+trainer.enable_progress_bar=false \
+trainer.log_every_n_steps=50

[2026-03-10 20:10:30,577][__main__][INFO] - 
Config:
user: single_user
dataset:
  train:
  - user: 89335547
    session: 2021-06-03-1622765527-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-02-1622681518-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-04-1622863166-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-22-1627003020-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-21-1626916256-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-22-1627004019-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-05-1622885888-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-02-1622679967-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f

In [60]:
TDS_GRU_AUG_BANDROT_RESULT = {
    "experiment_name": "tds_gru_aug_bandrot",
    "group": "augmentation",
    "model_type": "TDS+GRU",
    "config_path": "/home/bzhao29/emg2qwerty/config/model/tds_gru_ctc_aug_bandrot.yaml",
    "checkpoint_path": "/home/bzhao29/emg2qwerty/logs/2026-03-10/20-10-30/checkpoints/epoch=75-step=18240.ckpt",
    "preprocessing": "baseline_logspec",
    "augmentation": "band_rotation",
    "num_channels": 32,
    "train_fraction": 1.0,
    "sampling_rate": 2000,
    "epochs": 80,
    "val_CER": 15.706689834594727,
    "test_CER": 17.09531021118164,
    "DER": 1.5776960849761963,
    "IER": 2.7879836559295654,
    "SER": 12.729630470275879,
    "notes": "TDS+GRU with band rotation; hidden_size=384, num_layers=3, dropout=0.15, lr=3e-4, warmup=12, epochs=80",
    "status": "done",
}

add_or_replace_result(TDS_GRU_AUG_BANDROT_RESULT)

### Temporal jitter only

In [10]:
!LD_LIBRARY_PATH=$CONDA_PREFIX/lib python -m emg2qwerty.train \
user=single_user \
model=tds_gru_ctc_aug_jitter \
trainer.max_epochs=80 \
optimizer.lr=0.0003 \
lr_scheduler.scheduler.warmup_epochs=12 \
trainer.accelerator=gpu \
trainer.devices=1 \
num_workers=4 \
batch_size=16 \
+trainer.enable_progress_bar=false \
+trainer.log_every_n_steps=50

[2026-03-10 22:34:27,117][__main__][INFO] - 
Config:
user: single_user
dataset:
  train:
  - user: 89335547
    session: 2021-06-03-1622765527-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-02-1622681518-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-04-1622863166-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-22-1627003020-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-21-1626916256-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-22-1627004019-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-05-1622885888-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-02-1622679967-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f

In [61]:
TDS_GRU_AUG_JITTER_RESULT = {
    "experiment_name": "tds_gru_aug_jitter",
    "group": "augmentation",
    "model_type": "TDS+GRU",
    "config_path": "/home/bzhao29/emg2qwerty/config/model/tds_gru_ctc_aug_jitter.yaml",
    "checkpoint_path": "/home/bzhao29/emg2qwerty/logs/2026-03-10/22-34-27/checkpoints/epoch=78-step=18960.ckpt",
    "preprocessing": "baseline_logspec",
    "augmentation": "temporal_jitter",
    "num_channels": 32,
    "train_fraction": 1.0,
    "sampling_rate": 2000,
    "epochs": 80,
    "val_CER": 12.649535179138184,
    "test_CER": 14.069591522216797,
    "DER": 1.4912470579147339,
    "IER": 2.312513589859009,
    "SER": 10.265830993652344,
    "notes": "TDS+GRU with temporal jitter; hidden_size=384, num_layers=3, dropout=0.15, lr=3e-4, warmup=12, epochs=80",
    "status": "done",
}

add_or_replace_result(TDS_GRU_AUG_JITTER_RESULT)

### SpecAugment only

In [11]:
!LD_LIBRARY_PATH=$CONDA_PREFIX/lib python -m emg2qwerty.train \
user=single_user \
model=tds_gru_ctc_aug_specaug \
trainer.max_epochs=80 \
optimizer.lr=0.0003 \
lr_scheduler.scheduler.warmup_epochs=12 \
trainer.accelerator=gpu \
trainer.devices=1 \
num_workers=4 \
batch_size=16 \
+trainer.enable_progress_bar=false \
+trainer.log_every_n_steps=50

[2026-03-11 01:04:30,228][__main__][INFO] - 
Config:
user: single_user
dataset:
  train:
  - user: 89335547
    session: 2021-06-03-1622765527-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-02-1622681518-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-04-1622863166-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-22-1627003020-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-21-1626916256-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-22-1627004019-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-05-1622885888-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-02-1622679967-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f

In [62]:
TDS_GRU_AUG_SPECAUG_RESULT = {
    "experiment_name": "tds_gru_aug_specaug",
    "group": "augmentation",
    "model_type": "TDS+GRU",
    "config_path": "/home/bzhao29/emg2qwerty/config/model/tds_gru_ctc_aug_specaug.yaml",
    "checkpoint_path": "/home/bzhao29/emg2qwerty/logs/2026-03-11/01-04-30/checkpoints/epoch=77-step=18720.ckpt",
    "preprocessing": "baseline_logspec",
    "augmentation": "specaugment",
    "num_channels": 32,
    "train_fraction": 1.0,
    "sampling_rate": 2000,
    "epochs": 80,
    "val_CER": 14.842711448669434,
    "test_CER": 15.712124824523926,
    "DER": 1.448022484779358,
    "IER": 2.550248622894287,
    "SER": 11.71385383605957,
    "notes": "TDS+GRU with SpecAugment; hidden_size=384, num_layers=3, dropout=0.15, lr=3e-4, warmup=12, epochs=80",
    "status": "done",
}

add_or_replace_result(TDS_GRU_AUG_SPECAUG_RESULT)

### Band rotation + SpecAugmentation

In [2]:
!LD_LIBRARY_PATH=$CONDA_PREFIX/lib python -m emg2qwerty.train \
user=single_user \
model=tds_gru_ctc_aug_band_spec \
trainer.max_epochs=80 \
optimizer.lr=0.0003 \
lr_scheduler.scheduler.warmup_epochs=12 \
trainer.accelerator=gpu \
trainer.devices=1 \
num_workers=4 \
batch_size=16 \
+trainer.enable_progress_bar=false \
+trainer.log_every_n_steps=50

[2026-03-12 23:04:29,518][__main__][INFO] - 
Config:
user: single_user
dataset:
  train:
  - user: 89335547
    session: 2021-06-03-1622765527-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-02-1622681518-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-04-1622863166-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-22-1627003020-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-21-1626916256-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-22-1627004019-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-05-1622885888-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-02-1622679967-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f

In [63]:
TDS_GRU_AUG_BANDSPEC = {
    "experiment_name": "tds_gru_aug_band+spec",
    "group": "augmentation",
    "model_type": "TDS+GRU",
    "config_path": "/home/bzhao29/emg2qwerty/config/model/tds_gru_ctc_aug_band_spec.yaml",
    "checkpoint_path": "/home/bzhao29/emg2qwerty/logs/2026-03-12/23-04-29/checkpoints/epoch=77-step=18720.ckpt",
    "preprocessing": "baseline_logspec",
    "augmentation": "band_rotation + specaugmentation",
    "num_channels": 32,
    "train_fraction": 1.0,
    "sampling_rate": 2000,
    "epochs": 80,
    "val_CER": 16.747896194458008,
    "test_CER": 17.160146713256836,
    "DER": 1.7289820909500122,
    "IER": 2.723146677017212,
    "SER": 12.70801830291748,
    "notes": "TDS+GRU with Band Rotation + SpecAugment; hidden_size=384, num_layers=3, dropout=0.15, lr=3e-4, warmup=12, epochs=80",
    "status": "done",
}

add_or_replace_result(TDS_GRU_AUG_BANDSPEC)

### Temporal Jitter + SpecAugmentation

In [3]:
!LD_LIBRARY_PATH=$CONDA_PREFIX/lib python -m emg2qwerty.train \
user=single_user \
model=tds_gru_ctc_aug_temp_spec \
trainer.max_epochs=80 \
optimizer.lr=0.0003 \
lr_scheduler.scheduler.warmup_epochs=12 \
trainer.accelerator=gpu \
trainer.devices=1 \
num_workers=4 \
batch_size=16 \
+trainer.enable_progress_bar=false \
+trainer.log_every_n_steps=50

[2026-03-13 00:19:00,984][__main__][INFO] - 
Config:
user: single_user
dataset:
  train:
  - user: 89335547
    session: 2021-06-03-1622765527-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-02-1622681518-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-04-1622863166-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-22-1627003020-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-21-1626916256-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-22-1627004019-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-05-1622885888-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-02-1622679967-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f

In [64]:
TDS_GRU_AUG_TEMPSPEC = {
    "experiment_name": "tds_gru_aug_temp+spec",
    "group": "augmentation",
    "model_type": "TDS+GRU",
    "config_path": "/home/bzhao29/emg2qwerty/config/model/tds_gru_ctc_aug_temp_spec.yaml",
    "checkpoint_path": "/home/bzhao29/emg2qwerty/logs/2026-03-13/00-19-00/checkpoints/epoch=79-step=19200.ckpt",
    "preprocessing": "baseline_logspec",
    "augmentation": "temp_jitter + specaugmentation",
    "num_channels": 32,
    "train_fraction": 1.0,
    "sampling_rate": 2000,
    "epochs": 80,
    "val_CER": 16.747896194458008,
    "test_CER": 17.160146713256836,
    "DER": 1.7289820909500122,
    "IER": 2.723146677017212,
    "SER": 12.70801830291748,
    "notes": "TDS+GRU with Temporal Jitter + SpecAugment; hidden_size=384, num_layers=3, dropout=0.15, lr=3e-4, warmup=12, epochs=80",
    "status": "done",
}

add_or_replace_result(TDS_GRU_AUG_TEMPSPEC)

### Full augmentation

In [1]:
!LD_LIBRARY_PATH=$CONDA_PREFIX/lib python -m emg2qwerty.train \
user=single_user \
model=tds_gru_ctc_aug_full \
trainer.max_epochs=80 \
optimizer.lr=0.0003 \
lr_scheduler.scheduler.warmup_epochs=12 \
trainer.accelerator=gpu \
trainer.devices=1 \
num_workers=4 \
batch_size=16 \
+trainer.enable_progress_bar=false \
+trainer.log_every_n_steps=50

[2026-03-11 05:55:50,887][__main__][INFO] - 
Config:
user: single_user
dataset:
  train:
  - user: 89335547
    session: 2021-06-03-1622765527-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-02-1622681518-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-04-1622863166-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-22-1627003020-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-21-1626916256-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-22-1627004019-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-05-1622885888-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-02-1622679967-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f

In [65]:
FULL_AUG_RESULT = {
    "experiment_name": "tds_gru_full_aug",
    "group": "augmentation",
    "model_type": "TDS+GRU",
    "config_path": "/home/bzhao29/emg2qwerty/config/model/tds_gru_ctc_aug_full.yaml",
    "checkpoint_path": "/home/bzhao29/emg2qwerty/logs/2026-03-11/05-55-50/checkpoints/epoch=76-step=18480.ckpt",
    "preprocessing": "baseline_logspec",
    "augmentation": "band_rotation+temporal_jitter+specaugment",
    "num_channels": 32,
    "train_fraction": 1.0,
    "sampling_rate": 2000,
    "epochs": 80,
    "val_CER": 13.934426307678223,
    "test_CER": 14.307326316833496,
    "DER": 1.2318997383117676,
    "IER": 2.226064443588257,
    "SER": 10.84936237335205,
    "notes": "Full augmentation with TDS+GRU. Best checkpoint at epoch 76.",
    "status": "done",
}

add_or_replace_result(FULL_AUG_RESULT)

# Experimentation With Different Number of Channels

Now we will test with different number of channels and see how it affects the CER.
We will test 32, 24, 16, 8 channels.

## 24 Channels

In [4]:
!LD_LIBRARY_PATH=$CONDA_PREFIX/lib python -m emg2qwerty.train \
user=single_user \
model=tds_gru_ctc_channel_sweep \
trainer.max_epochs=80 \
optimizer.lr=0.0003 \
lr_scheduler.scheduler.warmup_epochs=12 \
trainer.accelerator=gpu \
trainer.devices=1 \
num_workers=4 \
batch_size=16 \
seed=42 \
channel_select.num_channels_total=24 \
module.electrode_channels=12 \
module.in_features=396 \
checkpoint='"/home/bzhao29/emg2qwerty/logs/2026-03-11/10-35-28/checkpoints/epoch=12-step=3120.ckpt"' \
+trainer.enable_progress_bar=false \
+trainer.log_every_n_steps=50

[2026-03-11 11:01:48,056][__main__][INFO] - 
Config:
user: single_user
dataset:
  train:
  - user: 89335547
    session: 2021-06-03-1622765527-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-02-1622681518-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-04-1622863166-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-22-1627003020-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-21-1626916256-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-22-1627004019-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-05-1622885888-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-02-1622679967-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f

In [66]:
TDS_GRU_CH24_RESULT = {
    "experiment_name": "tds_gru_24channels",
    "group": "channel_count",
    "model_type": "TDS+GRU",
    "config_path": "/home/bzhao29/emg2qwerty/config/model/tds_gru_ctc_channel_sweep.yaml",
    "checkpoint_path": "/home/bzhao29/emg2qwerty/logs/2026-03-11/11-01-47/checkpoints/epoch=64-step=15600.ckpt",
    "preprocessing": "baseline_logspec",
    "augmentation": "band_rotation+temporal_jitter",
    "num_channels": 24,
    "train_fraction": 1.0,
    "sampling_rate": 2000,
    "epochs": 80,
    "val_CER": 14.798404693603516,
    "test_CER": 16.576616287231445,
    "DER": 1.2318997383117676,
    "IER": 2.85282039642334,
    "SER": 12.49189567565918,
    "notes": "TDS+GRU using 24 total channels (12 per arm).",
    "status": "done",
}

add_or_replace_result(TDS_GRU_CH24_RESULT)

## 16 Channels

In [1]:
!LD_LIBRARY_PATH=$CONDA_PREFIX/lib python -m emg2qwerty.train \
user=single_user \
model=tds_gru_ctc_channel_sweep \
trainer.max_epochs=80 \
optimizer.lr=0.0003 \
lr_scheduler.scheduler.warmup_epochs=12 \
trainer.accelerator=gpu \
trainer.devices=1 \
num_workers=4 \
batch_size=16 \
seed=42 \
channel_select.num_channels_total=16 \
module.electrode_channels=8 \
module.in_features=264 \
+trainer.enable_progress_bar=false \
+trainer.log_every_n_steps=50

[2026-03-11 16:09:51,637][__main__][INFO] - 
Config:
user: single_user
dataset:
  train:
  - user: 89335547
    session: 2021-06-03-1622765527-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-02-1622681518-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-04-1622863166-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-22-1627003020-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-21-1626916256-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-22-1627004019-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-05-1622885888-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-02-1622679967-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f

In [67]:
TDS_GRU_CH16_RESULT = {
    "experiment_name": "tds_gru_16channels",
    "group": "channel_count",
    "model_type": "TDS+GRU",
    "config_path": "/home/bzhao29/emg2qwerty/config/model/tds_gru_ctc_channel_sweep.yaml",
    "checkpoint_path": "/home/bzhao29/emg2qwerty/logs/2026-03-11/16-09-51/checkpoints/epoch=62-step=15120.ckpt",
    "preprocessing": "baseline_logspec",
    "augmentation": "band_rotation+temporal_jitter",
    "num_channels": 16,
    "train_fraction": 1.0,
    "sampling_rate": 2000,
    "epochs": 80,
    "val_CER": 18.298625946044922,
    "test_CER": 19.10525131225586,
    "DER": 1.6641452312469482,
    "IER": 3.5444133281707764,
    "SER": 13.896693229675293,
    "notes": "TDS+GRU using 16 total channels (8 per arm).",
    "status": "done",
}

add_or_replace_result(TDS_GRU_CH16_RESULT)

## 8 Channels

In [2]:
!LD_LIBRARY_PATH=$CONDA_PREFIX/lib python -m emg2qwerty.train \
user=single_user \
model=tds_gru_ctc_channel_sweep \
trainer.max_epochs=80 \
optimizer.lr=0.0003 \
lr_scheduler.scheduler.warmup_epochs=12 \
trainer.accelerator=gpu \
trainer.devices=1 \
num_workers=4 \
batch_size=16 \
seed=42 \
channel_select.num_channels_total=8 \
module.electrode_channels=4 \
module.in_features=132 \
+trainer.enable_progress_bar=false \
+trainer.log_every_n_steps=50

[2026-03-11 18:34:52,252][__main__][INFO] - 
Config:
user: single_user
dataset:
  train:
  - user: 89335547
    session: 2021-06-03-1622765527-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-02-1622681518-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-04-1622863166-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-22-1627003020-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-21-1626916256-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-22-1627004019-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-05-1622885888-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-02-1622679967-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f

In [68]:
TDS_GRU_CH8_RESULT = {
    "experiment_name": "tds_gru_8channels",
    "group": "channel_count",
    "model_type": "TDS+GRU",
    "config_path": "/home/bzhao29/emg2qwerty/config/model/tds_gru_ctc_channel_sweep.yaml",
    "checkpoint_path": "/home/bzhao29/emg2qwerty/logs/2026-03-11/18-34-52/checkpoints/epoch=76-step=18480.ckpt",
    "preprocessing": "baseline_logspec",
    "augmentation": "band_rotation+temporal_jitter",
    "num_channels": 8,
    "train_fraction": 1.0,
    "sampling_rate": 2000,
    "epochs": 80,
    "val_CER": 32.676116943359375,
    "test_CER": 34.471580505371094,
    "DER": 3.4147396087646484,
    "IER": 7.326561450958252,
    "SER": 23.73027801513672,
    "notes": "TDS+GRU using 8 total channels (4 per arm).",
    "status": "done",
}

add_or_replace_result(TDS_GRU_CH8_RESULT)

# Training Data vs. CER

Using our current best model, we will train with different size of training data and see if the model generalizes better.

Due to time and resource constraints, we will just use 40 epochs to see the general trend in the CER to see whether less data will provide any better accuracies.

### Testing with 100% training data

In [7]:
!LD_LIBRARY_PATH=$CONDA_PREFIX/lib python -m emg2qwerty.train \
user=single_user \
model=tds_gru_ctc_noaug \
trainer.max_epochs=40 \
optimizer.lr=0.0003 \
lr_scheduler.scheduler.warmup_epochs=10 \
trainer.accelerator=gpu \
trainer.devices=1 \
num_workers=4 \
batch_size=16 \
seed=42 \
+datamodule.train_fraction=1 \
+trainer.enable_progress_bar=false \
+trainer.log_every_n_steps=50

[2026-03-11 22:44:29,641][__main__][INFO] - 
Config:
user: single_user
dataset:
  train:
  - user: 89335547
    session: 2021-06-03-1622765527-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-02-1622681518-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-04-1622863166-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-22-1627003020-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-21-1626916256-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-22-1627004019-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-05-1622885888-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-02-1622679967-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f

In [69]:
PART4_100_RESULT = {
    "experiment_name": "tds_gru_trainfrac_100",
    "group": "data_amount",
    "model_type": "TDS+GRU",
    "config_path": "/home/bzhao29/emg2qwerty/config/model/tds_gru_ctc_noaug.yaml",
    "checkpoint_path": "/home/bzhao29/emg2qwerty/logs/2026-03-11/22-44-29/checkpoints/epoch=39-step=9600.ckpt",
    "preprocessing": "baseline_logspec",
    "augmentation": "band_rotation+temporal_jitter",
    "num_channels": 32,
    "train_fraction": 1.0,
    "sampling_rate": 2000,
    "epochs": 40,
    "val_CER": 14.95347785949707,
    "test_CER": 16.252431869506836,
    "DER": 1.6209206581115723,
    "IER": 2.6583099365234375,
    "SER": 11.973200798034668,
    "notes": "TDS+GRU, 100% training data, 40 epochs.",
    "status": "done",
}
add_or_replace_result(PART4_100_RESULT)

### Testing with 75% training data

In [4]:
!LD_LIBRARY_PATH=$CONDA_PREFIX/lib python -m emg2qwerty.train \
user=single_user \
model=tds_gru_ctc_noaug \
trainer.max_epochs=40 \
optimizer.lr=0.0003 \
lr_scheduler.scheduler.warmup_epochs=10 \
trainer.accelerator=gpu \
trainer.devices=1 \
num_workers=4 \
batch_size=16 \
seed=42 \
+datamodule.train_fraction=0.75 \
+trainer.enable_progress_bar=false \
+trainer.log_every_n_steps=50

[2026-03-11 20:50:04,004][__main__][INFO] - 
Config:
user: single_user
dataset:
  train:
  - user: 89335547
    session: 2021-06-03-1622765527-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-02-1622681518-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-04-1622863166-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-22-1627003020-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-21-1626916256-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-22-1627004019-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-05-1622885888-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-02-1622679967-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f

In [70]:
PART4_75_RESULT = {
    "experiment_name": "tds_gru_trainfrac_75",
    "group": "data_amount",
    "model_type": "TDS+GRU",
    "config_path": "/home/bzhao29/emg2qwerty/config/model/tds_gru_ctc_noaug.yaml",
    "checkpoint_path": "/home/bzhao29/emg2qwerty/logs/2026-03-11/20-50-03/checkpoints/epoch=38-step=7020.ckpt",
    "preprocessing": "baseline_logspec",
    "augmentation": "band_rotation+temporal_jitter",
    "num_channels": 32,
    "train_fraction": 0.75,
    "sampling_rate": 2000,
    "epochs": 40,
    "val_CER": 17.922019958496094,
    "test_CER": 19.018802642822266,
    "DER": 1.6209206581115723,
    "IER": 3.5876376628875732,
    "SER": 13.8102445602417,
    "notes": "TDS+GRU, 75% training data, 40 epochs.",
    "status": "done",
}
add_or_replace_result(PART4_75_RESULT)

### Testing with 50% training data

In [5]:
!LD_LIBRARY_PATH=$CONDA_PREFIX/lib python -m emg2qwerty.train \
user=single_user \
model=tds_gru_ctc_noaug \
trainer.max_epochs=40 \
optimizer.lr=0.0003 \
lr_scheduler.scheduler.warmup_epochs=10 \
trainer.accelerator=gpu \
trainer.devices=1 \
num_workers=4 \
batch_size=16 \
seed=42 \
+datamodule.train_fraction=0.5 \
+trainer.enable_progress_bar=false \
+trainer.log_every_n_steps=50

[2026-03-11 21:45:27,939][__main__][INFO] - 
Config:
user: single_user
dataset:
  train:
  - user: 89335547
    session: 2021-06-03-1622765527-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-02-1622681518-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-04-1622863166-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-22-1627003020-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-21-1626916256-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-22-1627004019-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-05-1622885888-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-02-1622679967-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f

In [71]:
PART4_50_RESULT = {
    "experiment_name": "tds_gru_trainfrac_50",
    "group": "data_amount",
    "model_type": "TDS+GRU",
    "config_path": "/home/bzhao29/emg2qwerty/config/model/tds_gru_ctc_noaug.yaml",
    "checkpoint_path": "/home/bzhao29/emg2qwerty/logs/2026-03-11/21-45-27/checkpoints/epoch=35-step=4320.ckpt",
    "preprocessing": "baseline_logspec",
    "augmentation": "band_rotation+temporal_jitter",
    "num_channels": 32,
    "train_fraction": 0.50,
    "sampling_rate": 2000,
    "epochs": 40,
    "val_CER": 25.852901458740234,
    "test_CER": 25.54570960998535,
    "DER": 2.4854116439819336,
    "IER": 5.3814568519592285,
    "SER": 17.67884063720703,
    "notes": "TDS+GRU, 50% training data, 40 epochs.",
    "status": "done",
}
add_or_replace_result(PART4_50_RESULT)

### Testing with 25% training data

In [6]:
!LD_LIBRARY_PATH=$CONDA_PREFIX/lib python -m emg2qwerty.train \
user=single_user \
model=tds_gru_ctc_noaug \
trainer.max_epochs=40 \
optimizer.lr=0.0003 \
lr_scheduler.scheduler.warmup_epochs=10 \
trainer.accelerator=gpu \
trainer.devices=1 \
num_workers=4 \
batch_size=16 \
seed=42 \
+datamodule.train_fraction=0.25 \
+trainer.enable_progress_bar=false \
+trainer.log_every_n_steps=50

[2026-03-11 22:23:37,753][__main__][INFO] - 
Config:
user: single_user
dataset:
  train:
  - user: 89335547
    session: 2021-06-03-1622765527-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-02-1622681518-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-04-1622863166-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-22-1627003020-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-21-1626916256-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-22-1627004019-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-05-1622885888-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-02-1622679967-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f

In [72]:
PART4_25_RESULT = {
    "experiment_name": "tds_gru_trainfrac_25",
    "group": "data_amount",
    "model_type": "TDS+GRU",
    "config_path": "/home/bzhao29/emg2qwerty/config/model/tds_gru_ctc_noaug.yaml",
    "checkpoint_path": "/home/bzhao29/emg2qwerty/logs/2026-03-11/22-23-37/checkpoints/epoch=37-step=2280.ckpt",
    "preprocessing": "baseline_logspec",
    "augmentation": "band_rotation+temporal_jitter",
    "num_channels": 32,
    "train_fraction": 0.25,
    "sampling_rate": 2000,
    "epochs": 40,
    "val_CER": 99.60124206542969,
    "test_CER": 100.0,
    "DER": 0.0,
    "IER": 100.0,
    "SER": 0.0,
    "notes": "TDS+GRU, 25% training data, 40 epochs. Model failed to learn meaningful decoding.",
    "status": "done",
}
add_or_replace_result(PART4_25_RESULT)

# Sampling Frequency vs. CER

Lastly, we will test how different sampling frequency will affect the CER.

## 2000Hz

In [2]:
!LD_LIBRARY_PATH=$CONDA_PREFIX/lib python -m emg2qwerty.train \
user=single_user \
model=tds_gru_ctc_sampling_sweep \
downsample.factor=1 \
trainer.max_epochs=40 \
optimizer.lr=0.0003 \
lr_scheduler.scheduler.warmup_epochs=10 \
trainer.accelerator=gpu \
trainer.devices=1 \
num_workers=4 \
batch_size=16 \
seed=42 \
+trainer.enable_progress_bar=false \
+trainer.log_every_n_steps=50

[2026-03-12 04:38:20,776][__main__][INFO] - 
Config:
user: single_user
dataset:
  train:
  - user: 89335547
    session: 2021-06-03-1622765527-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-02-1622681518-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-04-1622863166-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-22-1627003020-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-21-1626916256-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-22-1627004019-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-05-1622885888-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-02-1622679967-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f

In [73]:
TDS_GRU_SR_FACTOR1_RESULT = {
    "experiment_name": "tds_gru_sampling_2000Hz",
    "group": "sampling_rate",
    "model_type": "TDS+GRU",
    "config_path": "/home/bzhao29/emg2qwerty/config/model/tds_gru_ctc_sampling_sweep.yaml",
    "checkpoint_path": "/home/bzhao29/emg2qwerty/logs/2026-03-12/04-38-20/checkpoints/epoch=38-step=9360.ckpt",
    "preprocessing": "baseline_logspec",
    "augmentation": "band_rotation+temporal_jitter",
    "num_channels": 32,
    "train_fraction": 1.0,
    "sampling_rate": 2000,
    "downsample_factor": 1,
    "epochs": 40,
    "val_CER": 15.086398124694824,
    "test_CER": 15.496002197265625,
    "DER": 1.5776960849761963,
    "IER": 2.679922103881836,
    "SER": 11.238383293151855,
    "notes": "TDS+GRU setup, factor=1 (2000 Hz), 40 epochs.",
    "status": "done",
}

add_or_replace_result(TDS_GRU_SR_FACTOR1_RESULT)

## 1000Hz

In [23]:
!LD_LIBRARY_PATH=$CONDA_PREFIX/lib python -m emg2qwerty.train \
user=single_user \
model=tds_gru_ctc_sampling_sweep \
downsample.factor=2 \
trainer.max_epochs=40 \
optimizer.lr=0.0003 \
lr_scheduler.scheduler.warmup_epochs=10 \
trainer.accelerator=gpu \
trainer.devices=1 \
num_workers=4 \
batch_size=16 \
seed=42 \
+trainer.enable_progress_bar=false \
+trainer.log_every_n_steps=50

[2026-03-12 06:39:20,837][__main__][INFO] - 
Config:
user: single_user
dataset:
  train:
  - user: 89335547
    session: 2021-06-03-1622765527-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-02-1622681518-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-04-1622863166-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-22-1627003020-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-21-1626916256-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-22-1627004019-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-05-1622885888-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-02-1622679967-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f

In [74]:
TDS_GRU_SR_FACTOR2_RESULT = {
    "experiment_name": "tds_gru_sampling_1000Hz",
    "group": "sampling_rate",
    "model_type": "TDS+GRU",
    "config_path": "/home/bzhao29/emg2qwerty/config/model/tds_gru_ctc_sampling_sweep.yaml",
    "checkpoint_path": "/home/bzhao29/emg2qwerty/logs/2026-03-12/06-39-20/checkpoints/epoch=37-step=4560.ckpt",
    "preprocessing": "baseline_logspec",
    "augmentation": "band_rotation+temporal_jitter",
    "num_channels": 32,
    "train_fraction": 1.0,
    "sampling_rate": 1000,
    "downsample_factor": 2,
    "epochs": 40,
    "val_CER": 27.846698760986328,
    "test_CER": 21.547439575195312,
    "DER": 1.9667171239852905,
    "IER": 5.684028625488281,
    "SER": 13.896693229675293,
    "notes": "TDS+GRU setup, factor=2 (1000 Hz), 40 epochs.",
    "status": "done",
}

add_or_replace_result(TDS_GRU_SR_FACTOR2_RESULT)

## 500Hz

In [11]:
!LD_LIBRARY_PATH=$CONDA_PREFIX/lib python -m emg2qwerty.train \
user=single_user \
model=tds_gru_ctc_sampling_sweep \
downsample.factor=4 \
trainer.max_epochs=40 \
optimizer.lr=0.0003 \
lr_scheduler.scheduler.warmup_epochs=10 \
trainer.accelerator=gpu \
trainer.devices=1 \
num_workers=4 \
batch_size=16 \
seed=42 \
+trainer.enable_progress_bar=false \
+trainer.log_every_n_steps=50

[2026-03-12 06:11:59,829][__main__][INFO] - 
Config:
user: single_user
dataset:
  train:
  - user: 89335547
    session: 2021-06-03-1622765527-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-02-1622681518-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-04-1622863166-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-22-1627003020-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-21-1626916256-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-22-1627004019-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-05-1622885888-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-02-1622679967-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f

In [75]:
TDS_GRU_SR_FACTOR4_RESULT = {
    "experiment_name": "tds_gru_sampling_500Hz",
    "group": "sampling_rate",
    "model_type": "TDS+GRU",
    "config_path": "/home/bzhao29/emg2qwerty/config/model/tds_gru_ctc_sampling_sweep.yaml",
    "checkpoint_path": "/home/bzhao29/emg2qwerty/logs/2026-03-12/06-11-59/checkpoints/epoch=38-step=4680.ckpt",
    "preprocessing": "baseline_logspec",
    "augmentation": "band_rotation+temporal_jitter",
    "num_channels": 32,
    "train_fraction": 1.0,
    "sampling_rate": 500,
    "downsample_factor": 3,
    "epochs": 40,
    "val_CER": 61.40895080566406,
    "test_CER": 46.83380126953125,
    "DER": 0.45385777950286865,
    "IER": 29.954614639282227,
    "SER": 16.425329208374023,
    "notes": "TDS+GRU setup, factor=4 (500 Hz), 40 epochs.",
    "status": "done",
}

add_or_replace_result(TDS_GRU_SR_FACTOR4_RESULT)

# Training our best model with full epochs (150)

In [26]:
!LD_LIBRARY_PATH=$CONDA_PREFIX/lib python -m emg2qwerty.train \
user=single_user \
model=tds_gru_ctc_noaug \
train=true \
optimizer.lr=0.0003 \
trainer.max_epochs=150 \
lr_scheduler.scheduler.warmup_epochs=15 \
module.hidden_size=384 \
module.num_layers=3 \
module.dropout=0.2 \
trainer.accelerator=gpu \
trainer.devices=1 \
num_workers=8 \
batch_size=16 \
seed=42 \
+trainer.enable_progress_bar=false \
+trainer.log_every_n_steps=50

[2026-03-12 07:12:13,838][__main__][INFO] - 
Config:
user: single_user
dataset:
  train:
  - user: 89335547
    session: 2021-06-03-1622765527-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-02-1622681518-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-04-1622863166-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-22-1627003020-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-21-1626916256-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-22-1627004019-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-05-1622885888-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-02-1622679967-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f

In [76]:
TDS_GRU_150EPOCH_RESULT = {
    "experiment_name": "tds_gru_150epochs",
    "group": "training_length",
    "model_type": "TDS+GRU",
    "config_path": "/home/bzhao29/emg2qwerty/config/model/tds_gru_ctc_noaug.yaml",
    "checkpoint_path": "/home/bzhao29/emg2qwerty/logs/2026-03-12/07-12-13/checkpoints/epoch=131-step=31680.ckpt",
    "preprocessing": "baseline_logspec",
    "augmentation": "band_rotation+temporal_jitter",
    "num_channels": 32,
    "train_fraction": 1.0,
    "sampling_rate": 2000,
    "epochs": 150,
    "val_CER": 12.051395416259766,
    "test_CER": 13.680570602416992,
    "DER": 1.2967365980148315,
    "IER": 2.3989624977111816,
    "SER": 9.984871864318848,
    "notes": "Extended training (150 epochs) with hidden_size=384, num_layers=3, dropout=0.2, lr=3e-4, warmup=15.",
    "status": "done",

}

add_or_replace_result(TDS_GRU_150EPOCH_RESULT)

In [92]:
experiment_to_delete = "tds_conformer_ctc__official"

results_df = results_df[results_df["experiment_name"] != experiment_to_delete]

results_df.to_csv(RESULTS_CSV, index=False)

print("Deleted:", experiment_to_delete)

Deleted: tds_conformer_ctc__official


In [96]:
column_to_delete = "downsample_factor"

results_df = results_df.drop(columns=[column_to_delete])

results_df.to_csv(RESULTS_CSV, index=False)

print("Deleted column:", column_to_delete)

Deleted column: downsample_factor
